# Prompts in the Client

**Guide for this lesson.** Do the exercise in the **starter** `cli-project/mcp_client.py`; the finished version is in `cli-project-complete/mcp_client.py`.

The server offers a `format` prompt (last lesson); now the **client** consumes it with two methods — `list_prompts` (what prompts exist) and `get_prompt` (render one with arguments into ready-to-send messages). This is the last piece: after it, all three MCP primitives — **tools, resources, prompts** — are fully wired end to end.

## What to modify

**File:** `cli-project/mcp_client.py` — replace the two `TODO` stubs:

```python
async def list_prompts(self) -> list[types.Prompt]:
    result = await self.session().list_prompts()
    return result.prompts


async def get_prompt(self, prompt_name, args: dict[str, str]):
    result = await self.session().get_prompt(prompt_name, args)
    return result.messages
```

(No new imports needed — `types` is already imported.) This completes `mcp_client.py`: `list_tools`/`call_tool`, `read_resource`, and now `list_prompts`/`get_prompt` are all implemented.

## How it works

- **`list_prompts`** → `session.list_prompts()` → `result.prompts` (each with a name, description, and expected arguments) — the CLI turns these into slash-commands.
- **`get_prompt(name, args)`** → `session.get_prompt(name, args)` → `result.messages`. The `args` dict's keys must match the server prompt's parameters; the server passes them as keyword args to the prompt function, which interpolates them and returns the messages.
- Those returned messages are a **ready-made conversation** — the client feeds them straight to Claude (no extra prompt-writing on the client side).

So the arg flow is: client `get_prompt("format", {"doc_id": "report.pdf"})` → server `format_document(doc_id="report.pdf")` → interpolated `UserMessage` list → back to the client → to Claude.

## The CLI flow

`core/cli_chat.py` uses `list_prompts` to register slash-commands and `get_prompt` to expand one when chosen:

```
type "/"            -> available prompts appear as commands (e.g. /format)
pick /format        -> prompted for its args (which doc_id)
                    -> get_prompt("format", {"doc_id": "..."}) -> messages
                    -> sent to Claude
Claude              -> uses the edit_document tool to reformat, returns the result
```

Notice how prompts + tools + resources combine: a *prompt* kicks off the task, Claude uses a *tool* to do the edit, and you could have pulled the doc in via a *resource* mention. That's the full MCP surface working together.

## Testing — the `/format` command

End-to-end, so test in a **terminal**:

```powershell
.\.venv\Scripts\Activate.ps1
cd 01-building-with-the-claude-api\07-mcp\cli-project-complete
python main.py
```

- Type **`/`** → slash-commands appear (including `format`).
- Choose **`/format`**, pick a document (e.g. `plan.md`) → the server's tested prompt is sent, and Claude reformats the doc into Markdown via the `edit_document` tool, then returns the final version.

Run in `cli-project/` once you've added these two methods there.

## Where this leaves us

The project is **complete**: a custom MCP **server** (tools, resources, prompts over in-memory docs) and an MCP **client** wired into a CLI chatbot that can read/edit docs, `@`-mention document contents, and run `/format`. You've now built and connected both halves of the protocol.

Next: **MCP review** (concept recap), then the section quiz.